# Antibiotic Prescribing RAG — Demonstration

This notebook demonstrates the reusable Python modules using the
precomputed Gemini results stored in the repository.

In [2]:
import json
from IPython.display import display

from data.synthetic_cases import synthetic_cases
from src.config import Settings
from src.pharmaceutical import load_aifa_tables, match_results_to_pharmaceuticals
from src.presentation import DoctorViewRenderer

## Load saved results

The Gemini results are loaded from the precomputed JSON file so the notebook
does not need to call the LLM again.

In [3]:
settings = Settings()
results_path = settings.results_path

if not results_path.exists():
    raise FileNotFoundError(
        f"Results file not found: {results_path}"
    )

with open(results_path, "r", encoding="utf-8") as f:
    gemini_results = json.load(f)

print(f"{len(synthetic_cases)} synthetic cases loaded")
print(f"{len(gemini_results)} saved results loaded")
print(f"Results source: {results_path}")

11 synthetic cases loaded
11 saved results loaded
Results source: results/gemini_results.json


## Match results to pharmaceutical products

The saved clinical decisions are matched to the corresponding
pharmaceutical products from the AIFA data.

In [4]:
tables = load_aifa_tables()

gemini_results = match_results_to_pharmaceuticals(
    gemini_results,
    tables["classe_a_principio_attivo"],
)

print("Pharmaceutical matching completed.")

/usr/local/lib/python3.13/dist-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'www.aifa.gov.it'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'www.aifa.gov.it'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'drive.aifa.gov.it'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


Pharmaceutical matching completed.


## Prepare retrieval results

The retrieval evidence stored inside the saved results is extracted so that
the presentation layer can display the AIFA pages used by the model.

In [5]:
retrieval_results = {
    case_id: result.get("retrieval", {})
    for case_id, result in gemini_results.items()
}

print(f"Retrieval results available for {len(retrieval_results)} cases")

Retrieval results available for 11 cases


## Display doctor-facing results

The presentation module renders each synthetic patient case with:

- patient information
- clinical note
- antibiotic decision
- selected active ingredient
- reasoning
- missing information
- corresponding pharmaceutical products
- AIFA evidence and selected pages

In [6]:
renderer = DoctorViewRenderer(
    synthetic_cases,
    gemini_results,
    retrieval_results,
)

print("Doctor-facing presentation ready.")
print("\nDisplaying results...\n")

for case in synthetic_cases:
    case_id = case["case_id"]
    display(renderer.render(case_id))

Doctor-facing presentation ready.

Displaying results...

